# Dictionary literals, adding and removing keys, accessing and replacing values, traversing

**Subject:** Introduction to Problem Solving and Reasoning (IPSR)  
**Unit 1 (Python Basics), Topic 14**  
**How to use this notebook:** read each section, run every code cell (Shift + Enter), then change the examples and run them again. Try each exercise before you open its solution.

## 1. Learning goals

After this notebook you can:

* create a dictionary with a literal `{ }` and with `dict()`
* add, update and remove keys in several ways
* read a value safely with `[]` and with `get`
* loop over the keys, the values and the pairs of a dictionary
* build a dictionary with a comprehension, and count things with one
* avoid the classic traps: unhashable keys, `1` and `True` colliding, changing a dictionary while looping, shared defaults, aliasing

## 2. What is a dictionary?

A **dictionary** (`dict`) stores **key : value** pairs. You look up a value by its key, like looking up a word in a real dictionary.

* A key must be **hashable**. Strings, numbers and tuples of those are fine. Lists are not.
* Each key appears **once**. If you repeat a key, the last value wins.
* Since Python 3.7 a dictionary remembers **insertion order**.

In [1]:
student = {"name": "Asha", "age": 18, "branch": "CSE"}
print(student)
print(type(student), len(student))

{'name': 'Asha', 'age': 18, 'branch': 'CSE'}
<class 'dict'> 3


You can also build a dictionary with `dict()`. Keyword arguments become string keys. A list of pairs also works.

In [2]:
a = dict(name="Asha", age=18)
b = dict([("name", "Asha"), ("age", 18)])
c = dict(zip(["name", "age"], ["Asha", 18]))
print(a == b == c)
print(dict())     # empty dictionary

True
{}


### `{}` is a dictionary, not a set
Empty curly brackets make an empty **dict**. To make an empty set, use `set()`.

In [3]:
print(type({}))
print(type(set()))
print(type({1, 2}))        # curly brackets with single values make a set
print(type({1: "one"}))    # curly brackets with key : value make a dict

<class 'dict'>
<class 'set'>
<class 'set'>
<class 'dict'>


## 3. Accessing values

Use `d[key]`. If the key is missing you get a `KeyError`. The method `get` returns `None` (or a default you choose) instead.

In [4]:
student = {"name": "Asha", "age": 18, "branch": "CSE"}
print(student["name"])
print(student.get("age"))
print(student.get("phone"))              # missing key gives None
print(student.get("phone", "not given")) # or your own default

try:
    print(student["phone"])
except KeyError as e:
    print("KeyError:", e)

Asha
18
None
not given
KeyError: 'phone'


In [5]:
print("name" in student)       # 'in' tests keys
print("Asha" in student)       # values are not checked
print("Asha" in student.values())

True
False
True


## 4. Adding and replacing values

`d[key] = value` does two jobs. If the key is new, it **adds** the pair. If the key exists, it **replaces** the value.

In [6]:
student = {"name": "Asha", "age": 18}
student["branch"] = "CSE"      # new key: added
student["age"] = 19            # existing key: replaced
print(student)

{'name': 'Asha', 'age': 19, 'branch': 'CSE'}


`update` changes many keys at once. It accepts another dictionary, pairs, or keyword arguments.

In [7]:
student.update({"age": 20, "city": "Bhopal"})
student.update(branch="IT")
print(student)

{'name': 'Asha', 'age': 20, 'branch': 'IT', 'city': 'Bhopal'}


`setdefault(key, default)` returns the value if the key exists. If not, it **adds** the key with the default and returns it. It never replaces an existing value.

In [8]:
print(student.setdefault("age", 99))        # exists: nothing changes
print(student.setdefault("phone", "n/a"))   # new: added
print(student)

20
n/a
{'name': 'Asha', 'age': 20, 'branch': 'IT', 'city': 'Bhopal', 'phone': 'n/a'}


## 5. Removing keys

| Way | What it does |
|---|---|
| `del d[k]` | removes the key, `KeyError` if missing |
| `d.pop(k)` | removes the key and **returns** its value, `KeyError` if missing |
| `d.pop(k, default)` | same, but returns `default` if missing (no error) |
| `d.popitem()` | removes and returns the **last inserted** pair |
| `d.clear()` | removes everything |

In [9]:
d = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
del d["a"]
print(d)
print(d.pop("b"))
print(d.pop("zzz", "no such key"))
print(d.popitem())          # last pair added
print(d)
d.clear()
print(d)

{'b': 2, 'c': 3, 'd': 4, 'e': 5}
2
no such key
('e', 5)
{'c': 3, 'd': 4}
{}


In [10]:
try:
    del d["x"]
except KeyError as e:
    print("del on missing key -> KeyError:", e)

try:
    {}.popitem()
except KeyError as e:
    print("popitem on empty dict -> KeyError:", e)

del on missing key -> KeyError: 'x'
popitem on empty dict -> KeyError: 'popitem(): dictionary is empty'


## 6. Traversing (looping)

Looping over a dictionary directly gives its **keys**. Use `.values()` for values and `.items()` for `(key, value)` pairs.
The order is the **insertion order**, not sorted order.

In [11]:
marks = {"maths": 78, "physics": 64, "python": 91}

for k in marks:
    print("key:", k)

for v in marks.values():
    print("value:", v)

for subject, score in marks.items():
    print(subject, "->", score)

key: maths
key: physics
key: python
value: 78
value: 64
value: 91
maths -> 78
physics -> 64
python -> 91


To loop in sorted order, sort the keys (or the items) first. The dictionary itself does not change.

In [12]:
for subject in sorted(marks):
    print(subject, marks[subject])

print("--- by score, highest first")
for subject, score in sorted(marks.items(), key=lambda pair: pair[1], reverse=True):
    print(subject, score)

maths 78
physics 64
python 91
--- by score, highest first
python 91
maths 78
physics 64


In [13]:
ordered = {}
ordered["z"] = 1
ordered["a"] = 2
ordered["m"] = 3
print(list(ordered))      # insertion order: z, a, m (not alphabetical)
print(list(ordered.items()))

['z', 'a', 'm']
[('z', 1), ('a', 2), ('m', 3)]


## 7. Dictionary comprehension

Same idea as a list comprehension, with `key: value`.

In [14]:
squares = {n: n * n for n in range(1, 6)}
print(squares)

marks = {"maths": 78, "physics": 64, "python": 91}
passed = {s: m for s, m in marks.items() if m >= 70}
print(passed)

flipped = {m: s for s, m in marks.items()}     # swap keys and values
print(flipped)

{1: 1, 2: 4, 3: 9, 4: 16, 5: 25}
{'maths': 78, 'python': 91}
{78: 'maths', 64: 'physics', 91: 'python'}


## 8. Worked example: counting words

A dictionary is the natural tool for counting. The key is the word, the value is how many times we saw it.

In [15]:
text = "to be or not to be that is the question"
counts = {}
for word in text.split():
    counts[word] = counts.get(word, 0) + 1
print(counts)

most = max(counts, key=counts.get)
print("most common:", most, counts[most])

{'to': 2, 'be': 2, 'or': 1, 'not': 1, 'that': 1, 'is': 1, 'the': 1, 'question': 1}
most common: to 2


The standard library has a shortcut, `collections.Counter`, which is a dictionary made for counting.

In [16]:
from collections import Counter
print(Counter(text.split()).most_common(2))

[('to', 2), ('be', 2)]


## 9. Edge cases

### 9.1 Keys must be hashable
A list can change, so it cannot be a key. A tuple of fixed values can.

In [17]:
try:
    d = {[1, 2]: "list as key"}
except TypeError as e:
    print("TypeError:", e)

d = {(1, 2): "tuple as key"}
print(d[(1, 2)])

TypeError: unhashable type: 'list'
tuple as key


### 9.2 `1`, `1.0` and `True` are the same key
Python treats them as equal with equal hashes, so they collide. The first key stays, the last value wins.

In [18]:
d = {}
d[1] = "int"
d[1.0] = "float"
d[True] = "bool"
print(d)
print(len(d))

{1: 'bool'}
1


### 9.3 Repeated keys in a literal
No error. The last value silently wins.

In [19]:
print({"a": 1, "a": 2})

{'a': 2}


### 9.4 Changing the size while looping
Adding or removing keys during a loop raises `RuntimeError`. Loop over a **copy** (a list of the keys) instead.

In [20]:
d = {"a": 1, "b": 2, "c": 3}
try:
    for k in d:
        if d[k] == 2:
            del d[k]
except RuntimeError as e:
    print("RuntimeError:", e)

d = {"a": 1, "b": 2, "c": 3}
for k in list(d):          # list(d) is a separate copy of the keys
    if d[k] == 2:
        del d[k]
print(d)

RuntimeError: dictionary changed size during iteration
{'a': 1, 'c': 3}


Changing the **value** of an existing key during a loop is allowed, because the size does not change.

In [21]:
d = {"a": 1, "b": 2}
for k in d:
    d[k] = d[k] * 10
print(d)

{'a': 10, 'b': 20}


### 9.5 `dict.fromkeys` shares one value
`dict.fromkeys(keys, value)` uses the **same object** for every key. With a list, all keys share one list.

In [22]:
bad = dict.fromkeys(["x", "y"], [])
bad["x"].append(1)
print(bad)                 # y changed too!
print(bad["x"] is bad["y"])

good = {k: [] for k in ["x", "y"]}    # a new list per key
good["x"].append(1)
print(good)

print(dict.fromkeys(["x", "y"], 0))   # fine with numbers and strings (they cannot change)

{'x': [1], 'y': [1]}
True
{'x': [1], 'y': []}
{'x': 0, 'y': 0}


### 9.6 `setdefault` and a default that is created every time
`d.setdefault(k, [])` builds a new empty list on every call, even when the key exists. It still works, and is a neat way to group items.

In [23]:
words = ["apple", "bat", "ant", "ball", "cat"]
groups = {}
for w in words:
    groups.setdefault(w[0], []).append(w)
print(groups)

{'a': ['apple', 'ant'], 'b': ['bat', 'ball'], 'c': ['cat']}


### 9.7 Copy or alias?
`b = a` does **not** copy. Both names point to the same dictionary. Use `.copy()` for a new one. `.copy()` is **shallow**: inner lists are still shared.

In [24]:
a = {"x": 1}
b = a                  # alias
c = a.copy()           # real copy
b["x"] = 99
print(a, b, c)

nested = {"scores": [1, 2]}
shallow = nested.copy()
shallow["scores"].append(3)
print(nested)          # the inner list is shared

import copy
deep = copy.deepcopy(nested)
deep["scores"].append(4)
print(nested, deep)

{'x': 99} {'x': 99} {'x': 1}
{'scores': [1, 2, 3]}
{'scores': [1, 2, 3]} {'scores': [1, 2, 3, 4]}


## 10. Quick summary

| Task | Code |
|---|---|
| Create | `{"a": 1}`, `dict(a=1)`, `{}` (empty dict) |
| Read | `d[k]` (KeyError if missing), `d.get(k, default)` |
| Add or replace | `d[k] = v`, `d.update(...)`, `d.setdefault(k, v)` (add only) |
| Remove | `del d[k]`, `d.pop(k, default)`, `d.popitem()`, `d.clear()` |
| Test a key | `k in d` |
| Loop | `for k in d`, `d.values()`, `d.items()`, `sorted(d)` |
| Build | `{k: v for ... in ...}` |
| Copy | `d.copy()` (shallow), `copy.deepcopy(d)` |
| Traps | list as key, `1` / `1.0` / `True` collide, no resizing in a loop, `fromkeys` shares the value |

## 11. Exercises

### Exercise 1

Create a dictionary `book` with keys `title`, `author` and `year` (use any values). Then add a key `pages`, change `year` to 2020, and print the final dictionary.

In [25]:
# Write your answer here

<details>
<summary><b>Show solution 1</b></summary>

```python
book = {"title": "Python Basics", "author": "R. Rao", "year": 2018}
book["pages"] = 320
book["year"] = 2020
print(book)
```

</details>

### Exercise 2

Given `prices = {"tea": 15, "coffee": 25, "milk": 30}`, print the price of `"juice"` as `"not sold"` without raising an error. Then remove `"milk"` and print what `pop` returned.

*Hint:* `get` with a default, and `pop`.

In [26]:
# Write your answer here

<details>
<summary><b>Show solution 2</b></summary>

```python
prices = {"tea": 15, "coffee": 25, "milk": 30}
print(prices.get("juice", "not sold"))
removed = prices.pop("milk")
print(removed)
print(prices)
```

</details>

### Exercise 3

For `marks = {"maths": 78, "physics": 64, "python": 91, "chem": 55}` print each subject and mark on one line, in alphabetical order of subject. Then print the average mark.

In [27]:
# Write your answer here

<details>
<summary><b>Show solution 3</b></summary>

The average is (78 + 64 + 91 + 55) / 4 = 72.0.

```python
marks = {"maths": 78, "physics": 64, "python": 91, "chem": 55}
for subject in sorted(marks):
    print(subject, marks[subject])
print(sum(marks.values()) / len(marks))
```

</details>

### Exercise 4

Use a dictionary comprehension to make a new dictionary with only the subjects where the mark is below 70, from the same `marks` dictionary as Exercise 3.

In [28]:
# Write your answer here

<details>
<summary><b>Show solution 4</b></summary>

```python
marks = {"maths": 78, "physics": 64, "python": 91, "chem": 55}
weak = {s: m for s, m in marks.items() if m < 70}
print(weak)
```

</details>

### Exercise 5

Count how many times each letter appears in the word `"mississippi"` using a dictionary and `get`. Print the result.

In [29]:
# Write your answer here

<details>
<summary><b>Show solution 5</b></summary>

Expected: `{'m': 1, 'i': 4, 's': 4, 'p': 2}`.

```python
counts = {}
for ch in "mississippi":
    counts[ch] = counts.get(ch, 0) + 1
print(counts)
```

</details>

### Exercise 6

From `d = {"a": 1, "b": 2, "c": 3, "d": 4}` delete every key whose value is even, **while looping**. Make it work without a RuntimeError.

*Hint:* loop over `list(d)`.

In [30]:
# Write your answer here

<details>
<summary><b>Show solution 6</b></summary>

A simpler way is `d = {k: v for k, v in d.items() if v % 2 != 0}`.

```python
d = {"a": 1, "b": 2, "c": 3, "d": 4}
for k in list(d):
    if d[k] % 2 == 0:
        del d[k]
print(d)
```

</details>

### Exercise 7

**Edge case.** Make a dictionary that gives each of the keys `"x"`, `"y"`, `"z"` its own empty list. Append `5` to the list for `"x"` only and show that `"y"` and `"z"` are still empty.

In [31]:
# Write your answer here

<details>
<summary><b>Show solution 7</b></summary>

`dict.fromkeys(["x", "y", "z"], [])` would share one list, so all three would change.

```python
d = {k: [] for k in ["x", "y", "z"]}
d["x"].append(5)
print(d)
```

</details>

## 12. Where to read more

* Whirlwind Tour of Python, *Built-In Data Structures*: https://jakevdp.github.io/WhirlwindTourOfPython/06-built-in-data-structures.html
* Real Python, *Dictionaries in Python*: https://realpython.com/python-dicts/